**tahap0_cek_silang.py — Cek silang perhitungan Tahap 0 (tanpa menjalankan model)**

Tujuan: mencocokkan angka yang Anda hitung di Master Data (sheet "Tahap 0").

Jalankan di Google Colab. Tidak butuh file apa pun (data diketik dari Master Data).

In [ ]:
import numpy as np
from scipy import stats

TAHUN = np.arange(2015, 2026)
JW    = np.array([6413735, 6551294, 6644412, 7996959, 20520104, 19610135, 22849394.5,
                  25755726, 30542580, 38134536, 40695654], float)        # baris 2 (data asli)
HOTEL = np.array([1165, 1170, 1179, 1617, 1817, 1848, 1696, 1818, 1820, 2000, 2291], float)  # baris 5
ODTW  = np.array([165, 158, 171, 177, 189, 180, 170, 183, 201, 218, 201], float)             # baris 8
LAHAN = np.array([43037.00491, 44561.0784555117, 50223.957001398, 56624.3602289948,
                  63480.0442236269, 58962.0411105434, 59887.5260314565, 57709.075808398,
                  75624.403529316, 73511.574651116, 55029.5352961885])                    # baris 15
TM    = np.array([5698325, 6097324, 10899900, 10293319, 13363923, 4753098, 7379577,
                  9267217, 11417702, 11503540, 11954114], float)                          # baris 29
TPK   = np.array([34.89, 37.72, 45.08, 45.11, 45.34, 28.90, 27.48, 44.99, 44.77, 42.17, 38.07]) / 100  # baris 33
INV   = np.array([53.0896, 373.8424, 165.3915, 568.3341, 741.4483, 489.6932, 848.8042,
                  463.5886, 1175.2552, 712.2835, 1325.950399])                            # baris 34
TPG   = np.array([1.97, 2.11, 1.91, 1.93, 2.02, 2.05, 2.15, 2.08, 2.10, 2.06, 2.07])      # baris 63
KAMAR = np.array([22594, 23613, 26164, 32869, 35717, 36482, 34386, 36644, 36883, 52233, 49066], float)  # baris 66

LUAS = 317036.0
B_ODTW, B_KEP, B_LAHAN, ELAS, BATAS = 0.40, 0.35, 0.25, 0.3, 1.5
ODTW_REF = ODTW[-1]
KEP_REF  = JW[-1] / LUAS
RDDL_REF = 1 - LAHAN[-1] / LUAS
KPU      = KAMAR[-1] / HOTEL[-1]                                      # Rata-rata Kamar per Unit (O71)
M        = np.sum(TM / TPG) / np.sum(TPK * KAMAR)                     # Malam Tersedia per Kamar (O68)
i = {t: k for k, t in enumerate(TAHUN)}

def dtd(odtw, jw, lahan):
    """Daya Tarik Destinasi Wisata — persamaan Vensim setelah perbaikan C & D."""
    k_odtw  = B_ODTW * min(BATAS, (odtw / ODTW_REF) ** ELAS)
    k_kep   = B_KEP * min(1.0, KEP_REF / (jw / LUAS))
    k_lahan = B_LAHAN * (max(0.0, 1 - lahan / LUAS) / RDDL_REF)
    return k_odtw + k_kep + k_lahan, (k_odtw, k_kep, k_lahan)

print("=== 0.3  DTD per tahun (versi data, persamaan terkoreksi) ===")
for t in TAHUN:
    d, (a, b, c) = dtd(ODTW[i[t]], JW[i[t]], LAHAN[i[t]])
    print(f"{t}: DTD = {d:.6f}   (ODTW {a:.6f} | kepadatan {b:.6f} | lahan {c:.6f})")

print("\n=== 0.4  Penurunan LPE ===")
g = JW[i[2025]] / JW[i[2024]] - 1
dtd2024 = dtd(ODTW[i[2024]], JW[i[2024]], LAHAN[i[2024]])[0]
lpe_A, lpe_B = g / (1 - 0.75), g / (dtd2024 - 0.75)
print(f"g 2024→2025          = {g:.7f}")
print(f"Cara A: LPE = {lpe_A:.6f}  LPD = {0.75*lpe_A:.6f}   (mengasumsikan DTD = 1)")
print(f"Cara B: LPE = {lpe_B:.6f}  LPD = {0.75*lpe_B:.6f}   (DTD(2024) = {dtd2024:.6f})")
jw2026 = JW[-1] * (1 + 0.26864 * 1.0 - 0.20148)
print(f"Cek G3 (BASE, nilai sekarang): JW 2026 seharusnya ≈ {jw2026:,.0f}")

print("\n=== G3/G4  Rekonstruksi Euler dari output P5 (Vensim) ===")
P5 = {2016: 15066309.46, 2017: 16005802.44, 2018: 17019055.10}   # dari Uji_P5_-_Vensim.xlsx
for t in (2016, 2017):
    obs = P5[t + 1] / P5[t]
    d_t  = dtd(ODTW[i[t]],     P5[t],     LAHAN[i[t]])[0]
    d_t1 = dtd(ODTW[i[t + 1]], P5[t + 1], LAHAN[i[t + 1]])[0]
    print(f"{t}→{t+1}: teramati {obs:.6f} | pakai DTD(t) {1+0.26864*d_t-0.20148:.6f}"
          f" | pakai DTD(t+1) {1+0.26864*d_t1-0.20148:.6f}")

print("\n=== 0.7  Identitas Sensitivitas Konstruksi ===")
s = ((HOTEL[-1] - HOTEL[0]) + 0.05 * HOTEL[:-1].sum()) / np.sum(INV[1:] * np.maximum(0, TPK[1:] - 0.275))
print(f"Sensitivitas (Ambang 0,275; Demolisi 0,05) = {s:.6f}   (harus 2,313846)")

print("\n=== 0.8  CI 95% LOGEST → Laju Konversi Dasar ===")
y = LAHAN[1:]                                   # 2016–2025 (2015 imputasi tidak dipakai)
x = np.arange(1, len(y) + 1)                    # 1..10, sama dengan bawaan LOGEST
reg = stats.linregress(x, np.log(y))
b1, se, df = reg.slope, reg.stderr, len(y) - 2
t_krit = stats.t.ppf(0.975, df)
R_bar = np.mean(1 - y / LUAS)
k_hat = (np.exp(b1) - 1) / R_bar
k_L = max(0.0, (np.exp(b1 - t_krit * se) - 1) / R_bar)
k_U = (np.exp(b1 + t_krit * se) - 1) / R_bar
print(f"β1 = {b1:.6f}  se(β1) = {se:.6f}  df = {df}  t* = {t_krit:.4f}  R̄ = {R_bar:.6f}")
print(f"k̂  = {k_hat:.7f}   (cek identitas: harus 0,0436052)")
print(f"k_L = {k_L:.7f}   k_U = {k_U:.7f}")
if (np.exp(b1 - t_krit * se) - 1) <= 0:
    print("Catatan: batas bawah pertumbuhan ≤ 0, jadi k_L dipotong di 0.")

print("\n=== 0.9  Deret B(t): bias konstanta konversi pada P1 revisi ===")
print("tahun |   B(t)   = malam tersedia +    TPG    + kamar/unit")
for t in range(2016, 2026):
    k = i[t]
    rp_konst = (TM[k] / 2.07) / (HOTEL[k] * KPU * M)                  # konstanta model
    rp_aktual = (TM[k] / TPG[k]) / (KAMAR[k] * M)                     # baris 40 Master Data
    B = np.log(rp_konst / TPK[k])
    c1 = np.log(rp_aktual / TPK[k])
    c2 = np.log(TPG[k] / 2.07)
    c3 = np.log((KAMAR[k] / HOTEL[k]) / KPU)
    print(f"{t}  | {B:+.4f}  = {c1:+.4f}        {c2:+.4f}     {c3:+.4f}")

=== 0.3  DTD per tahun (versi data, persamaan terkoreksi) ===
2015: DTD = 0.988447   (ODTW 0.377004 | kepadatan 0.350000 | lahan 0.261443)
2016: DTD = 0.982122   (ODTW 0.372133 | kepadatan 0.350000 | lahan 0.259989)
2017: DTD = 0.985651   (ODTW 0.381066 | kepadatan 0.350000 | lahan 0.254585)
2018: DTD = 0.983507   (ODTW 0.385029 | kepadatan 0.350000 | lahan 0.248478)
2019: DTD = 0.984618   (ODTW 0.392681 | kepadatan 0.350000 | lahan 0.241937)
2020: DTD = 0.983223   (ODTW 0.386975 | kepadatan 0.350000 | lahan 0.246248)
2021: DTD = 0.975761   (ODTW 0.380396 | kepadatan 0.350000 | lahan 0.245365)
2022: DTD = 0.986342   (ODTW 0.388899 | kepadatan 0.350000 | lahan 0.247443)
2023: DTD = 0.980349   (ODTW 0.400000 | kepadatan 0.350000 | lahan 0.230349)
2024: DTD = 0.992227   (ODTW 0.409862 | kepadatan 0.350000 | lahan 0.232365)
2025: DTD = 1.000000   (ODTW 0.400000 | kepadatan 0.350000 | lahan 0.250000)

=== 0.4  Penurunan LPE ===
g 2024→2025          = 0.0671601
Cara A: LPE = 0.268640  LPD = 